# 04 — Análise dos Resultados

**Projeto:** Avaliação Automática de Dificuldade em Jogos de Plataforma 2D
**Disciplina:** Redes Neurais Artificiais — PPGCC/UNESP

Este notebook consolida os logs dos 36 treinos (3 algoritmos × 4 fases × 3 seeds)
em métricas agregadas, tabelas, figuras e testes estatísticos, alimentando as
seções *Experimentos e Resultados* e *Conclusões* do artigo.

**Premissa:** os 3 notebooks de treino (01, 02, 03) já foram executados e os
arquivos `*_eval.csv` em `logs/{algo}/` + os modelos finais em `models/{algo}/`
existem. Se faltar algo, este notebook ainda roda — apenas reportará menos linhas.

## 1. Setup

In [ ]:
# --- 1. Setup ---
import os, sys, csv, json
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import imageio.v2 as imageio

from scipy import stats

# RL stack (apenas para carregar modelos no render comparativo)
import torch
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv, WarpFrame
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecFrameStack, VecTransposeImage
)

# Visual
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["savefig.dpi"] = 200
PALETTE = {"DQN": "#1f77b4", "PPO": "#d62728", "A2C": "#2ca02c"}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Imports OK | device={DEVICE}")


## 2. Configuração

In [ ]:
# --- 2. Paths ---
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive", force_remount=False)
        BASE_DIR = Path("/content/drive/MyDrive/mario_drl_results")
    except Exception:
        BASE_DIR = Path("/content/mario_drl_results")
else:
    BASE_DIR = Path("./mario_drl_results").resolve()

MODELS_DIR = BASE_DIR / "models"
LOGS_DIR   = BASE_DIR / "logs"
FIGS_DIR   = BASE_DIR / "figures"
VIDS_DIR   = BASE_DIR / "videos"
for d in [FIGS_DIR, VIDS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STAGES = ["1-1", "1-2", "4-1", "8-1"]
SEEDS  = [42, 123, 2024]
ALGOS  = ["DQN", "PPO", "A2C"]
STAGE_LENGTH = {"1-1": 3266, "1-2": 3266, "4-1": 3866, "8-1": 3266}
TOTAL_TIMESTEPS = 500_000

print(f"BASE: {BASE_DIR}")
print(f"  logs   → {LOGS_DIR}")
print(f"  models → {MODELS_DIR}")
print(f"  figs   → {FIGS_DIR}")


## 3. Carregamento dos logs de avaliação

In [ ]:
# --- 3.1 Função de carregamento ---
def load_eval_logs(logs_root: Path) -> pd.DataFrame:
    """Lê todos os *_eval.csv em logs/{algo}/ e concatena em UM DataFrame.

    Schema esperado (linha = 1 episódio de avaliação):
      algo, stage, seed, timestep, ep_idx, ep_reward, ep_length,
      x_pos_max, flag_get, deaths, time_remaining, ts_iso
    """
    frames = []
    if not logs_root.exists():
        print(f"✗ pasta de logs não existe: {logs_root}")
        return pd.DataFrame()
    for algo_dir in sorted(logs_root.iterdir()):
        if not algo_dir.is_dir() or algo_dir.name == "tb":
            continue
        for csv_path in sorted(algo_dir.glob("*_eval.csv")):
            try:
                df = pd.read_csv(csv_path)
                # Robust types
                df["timestep"]  = df["timestep"].astype(int)
                df["seed"]      = df["seed"].astype(int)
                df["stage"]     = df["stage"].astype(str)
                df["flag_get"]  = df["flag_get"].astype(bool)
                df["ep_reward"] = df["ep_reward"].astype(float)
                df["x_pos_max"] = df["x_pos_max"].astype(int)
                df["deaths"]    = df["deaths"].astype(int)
                frames.append(df)
            except Exception as e:
                print(f"  ✗ {csv_path.name}: {e}")
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out = out.sort_values(["algo", "stage", "seed", "timestep", "ep_idx"]).reset_index(drop=True)
    return out

print("✓ load_eval_logs() definido")


In [ ]:
# --- 3.2 Carregar dados ---
df = load_eval_logs(LOGS_DIR)
if df.empty:
    print("⚠️  Nenhum log encontrado. Rode os notebooks de treino primeiro.")
else:
    print(f"✓ {len(df):,} linhas (episódios) | "
          f"{df['algo'].nunique()} algos × {df['stage'].nunique()} fases × "
          f"{df['seed'].nunique()} seeds")
    print(f"  Faixa de timesteps: {df['timestep'].min():,} → {df['timestep'].max():,}")
    print(f"  Algoritmos: {sorted(df['algo'].unique())}")
    print(f"  Fases:      {sorted(df['stage'].unique())}")
    print(f"  Seeds:      {sorted(df['seed'].unique())}")


## 4. Visão geral do dataset

In [ ]:
# --- 4. Visão geral ---
if df.empty:
    print("(sem dados — pulando)")
else:
    completude = (
        df.groupby(["algo", "stage", "seed"])
          .size().reset_index(name="n_eval_eps")
          .pivot_table(index=["algo", "stage"], columns="seed",
                       values="n_eval_eps", aggfunc="first", fill_value=0)
    )
    print("Episódios de avaliação registrados por (algo, stage, seed):")
    print(completude)

    print("\nRecompensa por episódio — descritiva:")
    print(df.groupby("algo")["ep_reward"].describe()[["count", "mean", "50%", "std", "max"]])


## 5. Métricas Grupo I — comparação entre arquiteturas

### 5.1 Definições (Grupo I)

Sejam $R(t)$ a recompensa média de avaliação no checkpoint de timestep $t$
e $T$ o total de timesteps de treino. Para cada $(\text{algo}, \text{stage}, \text{seed})$
computamos:

- $\bar{R}_{\text{final}}$: média de $R$ nos últimos 50 000 timesteps
  (estado convergido)
- AUC normalizada: $\frac{1}{R_{\max} \cdot T}\int_0^T R(t)\, dt$, com
  $R_{\max}$ o máximo observado naquele run — escala $[0,1]$, mede
  eficiência amostral
- $t_{80\%}$: primeiro timestep em que $R(t)$ atinge $0{,}8 R_{\max}$
  (em frações de $T$; valor $\to 1$ se nunca atingir)
- $\sigma_{\text{final}}$: desvio-padrão de $R$ nos últimos 50 000
  timesteps (estabilidade)

In [ ]:
# --- 5.1 Cálculo Grupo I ---
def compute_group1_metrics(df_run: pd.DataFrame, total_timesteps: int = TOTAL_TIMESTEPS,
                            window: int = 50_000) -> dict:
    """Para um único run (algo, stage, seed), retorna métricas Grupo I."""
    if df_run.empty:
        return dict(R_final=np.nan, AUC_norm=np.nan, t80=np.nan, sigma_final=np.nan)
    # Agrega múltiplos eps por checkpoint (timestep) usando mediana
    g = df_run.groupby("timestep")["ep_reward"].median().sort_index()
    ts = g.index.values.astype(float)
    R  = g.values.astype(float)

    R_max = float(R.max()) if R.size else 0.0

    # R_final: janela de últimos 50k timesteps
    mask_final = ts >= (total_timesteps - window)
    R_final = float(np.mean(R[mask_final])) if mask_final.any() else float(R[-1])

    # σ_final: desvio dentro da janela final (sobre episódios individuais)
    df_window = df_run[df_run["timestep"] >= (total_timesteps - window)]
    sigma_final = float(df_window["ep_reward"].std(ddof=1)) if len(df_window) > 1 else 0.0

    # AUC normalizada (regra do trapézio)
    # NumPy 2.0+ renomeou trapz → trapezoid; mantemos compatibilidade.
    _trapz = getattr(np, "trapezoid", None) or np.trapz
    if R_max > 0 and ts.max() > ts.min():
        auc = _trapz(R, ts) / (R_max * total_timesteps)
    else:
        auc = 0.0

    # t80: primeiro ts em que R(t) ≥ 0.8 * R_max (em frações de T)
    threshold = 0.8 * R_max
    above = np.where(R >= threshold)[0]
    if above.size and R_max > 0:
        t80 = float(ts[above[0]]) / total_timesteps
    else:
        t80 = 1.0   # nunca atingiu

    return dict(R_final=R_final, AUC_norm=float(auc), t80=t80, sigma_final=sigma_final)


# Aplica em todos os runs (algo, stage, seed) → DataFrame de métricas
def build_group1_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (algo, stage, seed), sub in df.groupby(["algo", "stage", "seed"]):
        m = compute_group1_metrics(sub)
        rows.append(dict(algo=algo, stage=stage, seed=seed, **m))
    return pd.DataFrame(rows)

if not df.empty:
    g1_run = build_group1_table(df)
    print(f"✓ Métricas Grupo I computadas para {len(g1_run)} runs")
    print(g1_run.head(8).to_string(index=False))
else:
    g1_run = pd.DataFrame()


## 5.2 Tabela Grupo I (mediana ± IQR sobre 3 seeds)

In [ ]:
# --- 5.2 Tabela agregada (mediana ± IQR sobre as 3 seeds) ---
def aggregate_iqr(series: pd.Series) -> str:
    if len(series) == 0 or series.isna().all():
        return "—"
    med = float(series.median())
    q1, q3 = float(series.quantile(0.25)), float(series.quantile(0.75))
    return f"{med:.2f} (IQR {q1:.2f}–{q3:.2f})"

if not g1_run.empty:
    g1_agg = (
        g1_run
        .groupby(["algo", "stage"])
        .agg({"R_final":   aggregate_iqr,
              "AUC_norm":  aggregate_iqr,
              "t80":       aggregate_iqr,
              "sigma_final": aggregate_iqr})
        .reset_index()
    )
    print("Tabela Grupo I — mediana (IQR) sobre 3 seeds:")
    print(g1_agg.to_string(index=False))
    g1_agg.to_csv(FIGS_DIR / "table_group1.csv", index=False)
    print(f"\n✓ exportado: {FIGS_DIR / 'table_group1.csv'}")


## 5.3 Curvas de aprendizado

In [ ]:
# --- 5.3 Curvas de aprendizado (1 subplot por fase, 3 linhas algos) ---
if df.empty:
    print("(sem dados)")
else:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
    axes = axes.flatten()
    for i, stage in enumerate(STAGES):
        ax = axes[i]
        for algo in ALGOS:
            sub = df[(df["algo"] == algo) & (df["stage"] == stage)]
            if sub.empty:
                continue
            # Mediana e IQR por timestep sobre seeds × ep_idx
            agg = sub.groupby("timestep")["ep_reward"].agg(["median", "min", "max"]).reset_index()
            agg["q1"] = sub.groupby("timestep")["ep_reward"].quantile(0.25).values
            agg["q3"] = sub.groupby("timestep")["ep_reward"].quantile(0.75).values
            ax.plot(agg["timestep"], agg["median"], color=PALETTE[algo],
                    label=algo, linewidth=2.0)
            ax.fill_between(agg["timestep"], agg["q1"], agg["q3"],
                            color=PALETTE[algo], alpha=0.18)
        ax.set_title(f"Fase {stage}", fontsize=11)
        ax.set_xlabel("Timesteps de treino")
        if i % 2 == 0:
            ax.set_ylabel("Recompensa de avaliação (mediana)")
        ax.legend(loc="lower right", frameon=False, fontsize=9)
        ax.grid(alpha=0.3)
    fig.suptitle("Curvas de aprendizado — mediana ± IQR (3 seeds × 5 eps det.)",
                 fontsize=12, y=1.00)
    fig.tight_layout()
    out = FIGS_DIR / "learning_curves.png"
    fig.savefig(out); plt.show()
    print(f"✓ salvo: {out}")


## 5.4 Barras comparativas Grupo I

In [ ]:
# --- 5.4 Barras comparativas (R_final, AUC, t80, σ_final) ---
if not g1_run.empty:
    metric_titles = {
        "R_final":     "Recompensa final (média últimos 50k)",
        "AUC_norm":    "AUC normalizada [0,1]",
        "t80":         "Tempo para 80% do máx (frac. de T)",
        "sigma_final": "Desvio-padrão da recompensa final",
    }
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
    axes = axes.flatten()
    for i, (metric, title) in enumerate(metric_titles.items()):
        ax = axes[i]
        sns.boxplot(data=g1_run, x="stage", y=metric, hue="algo",
                    order=STAGES, hue_order=ALGOS, palette=PALETTE, ax=ax)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Fase"); ax.set_ylabel("")
        ax.legend(title="", frameon=False, fontsize=8, loc="best")
        ax.grid(alpha=0.3)
    fig.suptitle("Métricas Grupo I — distribuição sobre 3 seeds",
                 fontsize=12, y=1.00)
    fig.tight_layout()
    out = FIGS_DIR / "group1_bars.png"
    fig.savefig(out); plt.show()
    print(f"✓ salvo: {out}")


## 6. Métricas Grupo II — dificuldade

### 6.1 Definições (Grupo II)

Considerando os episódios determinísticos do **último checkpoint** (proxy do
agente convergido), para cada $(\text{algo}, \text{stage}, \text{seed})$:

- $\tau$: taxa de conclusão $= \text{média}(\mathbb{1}\{\text{flag\_get}\})$
- $\bar{d}$: distância normalizada $= \frac{\text{max\_x\_pos}}{\text{stage\_length}}$
- Mortes por episódio (mediana)
- Tempo em frames consumido apenas nos episódios bem-sucedidos
  (proxy da "limpeza" da execução)

Comprimentos canônicos (em pixels), retirados do disassembly do SMB NES:
1-1: 3266 · 1-2: 3266 · 4-1: 3866 · 8-1: 3266

In [ ]:
# --- 6.1 Cálculo Grupo II ---
def last_checkpoint_episodes(df_run: pd.DataFrame) -> pd.DataFrame:
    """Filtra para apenas o último timestep de avaliação registrado."""
    if df_run.empty:
        return df_run
    last_ts = df_run["timestep"].max()
    return df_run[df_run["timestep"] == last_ts].copy()


def compute_group2_metrics(df_run: pd.DataFrame, stage: str) -> dict:
    last = last_checkpoint_episodes(df_run)
    if last.empty:
        return dict(tau=np.nan, d_bar=np.nan, deaths_med=np.nan,
                    time_success=np.nan, n_eps=0)
    L = STAGE_LENGTH[stage]
    tau   = float(last["flag_get"].mean())
    d_bar = float(last["x_pos_max"].mean()) / L
    deaths_med = float(last["deaths"].median())
    # Tempo (frames) em runs bem-sucedidos
    succ = last[last["flag_get"]]
    if len(succ):
        # time_remaining é o tempo NO HUD restante; tempo CONSUMIDO ~ 400 - time_remaining
        # mas o valor inicial do HUD varia por fase (geralmente 400 ou 300).
        # Usamos ep_length como proxy estável (steps de policy × 4 frames de skip).
        time_success = float(succ["ep_length"].median() * 4)
    else:
        time_success = float("nan")
    return dict(tau=tau, d_bar=d_bar, deaths_med=deaths_med,
                time_success=time_success, n_eps=int(len(last)))


def build_group2_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (algo, stage, seed), sub in df.groupby(["algo", "stage", "seed"]):
        m = compute_group2_metrics(sub, stage)
        rows.append(dict(algo=algo, stage=stage, seed=seed, **m))
    return pd.DataFrame(rows)

if not df.empty:
    g2_run = build_group2_table(df)
    print(f"✓ Métricas Grupo II computadas para {len(g2_run)} runs")
    print(g2_run.head(8).to_string(index=False))
else:
    g2_run = pd.DataFrame()


## 6.2 Tabela Grupo II

In [ ]:
# --- 6.2 Tabela Grupo II (mediana ± IQR) ---
if not g2_run.empty:
    g2_agg = (
        g2_run
        .groupby(["algo", "stage"])
        .agg({"tau":          aggregate_iqr,
              "d_bar":        aggregate_iqr,
              "deaths_med":   aggregate_iqr,
              "time_success": aggregate_iqr})
        .reset_index()
    )
    print("Tabela Grupo II — mediana (IQR) sobre 3 seeds:")
    print(g2_agg.to_string(index=False))
    g2_agg.to_csv(FIGS_DIR / "table_group2.csv", index=False)
    print(f"\n✓ exportado: {FIGS_DIR / 'table_group2.csv'}")


## 6.3 Dificuldade por fase (boxplots)

In [ ]:
# --- 6.3 Boxplots de dificuldade por fase (1 subplot por métrica) ---
if not g2_run.empty:
    metric_titles = {
        "tau":          "Taxa de conclusão τ",
        "d_bar":        "Distância normalizada d̄",
        "deaths_med":   "Mortes por episódio (mediana)",
        "time_success": "Tempo (frames) em episódios bem-sucedidos",
    }
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
    axes = axes.flatten()
    for i, (metric, title) in enumerate(metric_titles.items()):
        ax = axes[i]
        sns.boxplot(data=g2_run, x="stage", y=metric, hue="algo",
                    order=STAGES, hue_order=ALGOS, palette=PALETTE, ax=ax)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Fase"); ax.set_ylabel("")
        ax.legend(title="", frameon=False, fontsize=8, loc="best")
        ax.grid(alpha=0.3)
    fig.suptitle("Métricas Grupo II — dificuldade por fase",
                 fontsize=12, y=1.00)
    fig.tight_layout()
    out = FIGS_DIR / "group2_boxplots.png"
    fig.savefig(out); plt.show()
    print(f"✓ salvo: {out}")


## 7. Análise Estatística

### 7.1 Spearman — correlação com a dificuldade canônica

Ordenamos as fases pela dificuldade **canônica** do jogo (progressão 1-1 → 1-2 → 4-1 → 8-1)
e comparamos com o ranking produzido pelas métricas do agente.

Para cada $(\text{algo}, \text{métrica})$, computamos $\rho$ de Spearman entre
o ranking canônico das 4 fases e o ranking induzido pela mediana da métrica sobre
as 3 seeds. Métricas de "facilidade" (τ, d̄) devem produzir $\rho$ negativo (fase mais
difícil → menor valor). Métricas de "dificuldade" (mortes) devem produzir $\rho$ positivo.

A força esperada da correlação é o eixo central do paper: $|\rho| \geq 0{,}8$ valida
a hipótese de que as métricas refletem a progressão canônica.

In [ ]:
# --- 7.1 Spearman ---
# Ranking canônico das fases (1 = mais fácil, 4 = mais difícil)
RANK_CANONICAL = {"1-1": 1, "1-2": 2, "4-1": 3, "8-1": 4}
SIGNS = {"tau": -1, "d_bar": -1, "deaths_med": +1, "time_success": +1}
#         ↑ esperamos que tau e d_bar DECRESÇAM com dificuldade, deaths AUMENTE.

def spearman_per_algo(g2_run: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for algo in ALGOS:
        sub = g2_run[g2_run["algo"] == algo]
        if sub.empty:
            continue
        med = (sub.groupby("stage")
                 .agg({"tau": "median", "d_bar": "median",
                       "deaths_med": "median", "time_success": "median"}))
        # Garante a ordem canônica
        med = med.reindex(STAGES)
        canonical = np.array([RANK_CANONICAL[s] for s in med.index])
        for metric in ["tau", "d_bar", "deaths_med", "time_success"]:
            vals = med[metric].values
            mask = ~np.isnan(vals)
            if mask.sum() < 3:
                rows.append(dict(algo=algo, metric=metric,
                                 rho=np.nan, p=np.nan, n=int(mask.sum())))
                continue
            rho, p = stats.spearmanr(canonical[mask], vals[mask])
            rows.append(dict(algo=algo, metric=metric,
                             rho=float(rho), p=float(p), n=int(mask.sum())))
    return pd.DataFrame(rows)

if not g2_run.empty:
    sp = spearman_per_algo(g2_run)
    print("Spearman ρ entre rank canônico das fases e mediana da métrica:")
    print(sp.to_string(index=False))
    sp.to_csv(FIGS_DIR / "spearman.csv", index=False)
    print(f"\n✓ exportado: {FIGS_DIR / 'spearman.csv'}")
    print("\nInterpretação: τ e d̄ esperam-se negativos (fase mais difícil → métrica menor);")
    print("              mortes esperam-se positivas; |ρ|≥0.8 valida a hipótese H1.")


### 7.2 Mann-Whitney U — comparação pareada de algoritmos

Para cada fase, comparamos pareadamente os 3 algoritmos (DQN×PPO, DQN×A2C, PPO×A2C)
na recompensa final dos episódios do último checkpoint. Mann-Whitney U é o
teste não-paramétrico apropriado: não assume normalidade, lida com pequenas
amostras (3 seeds × 5 eps = 15 valores por algoritmo) e é o padrão de fato
em comparações de RL.

In [ ]:
# --- 7.2 Mann-Whitney U pareado por fase ---
def mann_whitney_pairwise(df: pd.DataFrame, metric: str = "ep_reward") -> pd.DataFrame:
    rows = []
    for stage in STAGES:
        for a, b in combinations(ALGOS, 2):
            xa = df[(df["algo"] == a) & (df["stage"] == stage)][metric].values
            xb = df[(df["algo"] == b) & (df["stage"] == stage)][metric].values
            if len(xa) < 3 or len(xb) < 3:
                rows.append(dict(stage=stage, algo_a=a, algo_b=b,
                                 n_a=len(xa), n_b=len(xb),
                                 U=np.nan, p=np.nan,
                                 median_a=np.nan, median_b=np.nan))
                continue
            U, p = stats.mannwhitneyu(xa, xb, alternative="two-sided")
            rows.append(dict(
                stage=stage, algo_a=a, algo_b=b,
                n_a=int(len(xa)), n_b=int(len(xb)),
                U=float(U), p=float(p),
                median_a=float(np.median(xa)), median_b=float(np.median(xb)),
            ))
    return pd.DataFrame(rows)

if not df.empty:
    # Considera apenas episódios do último checkpoint de cada run (mais estável)
    last_eps = df.loc[df.groupby(["algo", "stage", "seed"])["timestep"].transform("max")
                      == df["timestep"]]
    mw = mann_whitney_pairwise(last_eps, metric="ep_reward")
    mw["sig"] = mw["p"].apply(lambda p: "***" if p < 0.001 else
                              "**" if p < 0.01 else
                              "*" if p < 0.05 else "ns")
    print("Mann-Whitney U (ep_reward, último checkpoint):")
    print(mw.to_string(index=False))
    mw.to_csv(FIGS_DIR / "mann_whitney.csv", index=False)
    print(f"\n✓ exportado: {FIGS_DIR / 'mann_whitney.csv'}")


## 8. Comparação visual lado-a-lado (DQN × PPO × A2C)

In [ ]:
# --- 8. GIF lado-a-lado: DQN × PPO × A2C no mesmo seed/fase ---
# Mesmo seed de avaliação → mesmo estado inicial → comparação justa.

def _ensure_cv2():
    """cv2 é usado apenas em renders comparativos. Lazy import + auto-install."""
    try:
        import cv2 as _cv2
        return _cv2
    except ImportError:
        import subprocess
        subprocess.run("pip install --quiet opencv-python-headless",
                       shell=True, check=True)
        import cv2 as _cv2
        return _cv2


def _build_env_for_render(stage: str, seed: int):
    raw_holder = {}
    def _thunk():
        env = gym_super_mario_bros.make(
            f"SuperMarioBros-{stage}-v0", render_mode="rgb_array")
        env = JoypadSpace(env, SIMPLE_MOVEMENT)
        env = MaxAndSkipEnv(env, skip=4)
        raw_holder["env"] = env
        env = WarpFrame(env, width=84, height=84)
        env = Monitor(env)
        return env
    venv = DummyVecEnv([_thunk])
    venv = VecFrameStack(venv, n_stack=4, channels_order="last")
    venv = VecTransposeImage(venv)
    venv.seed(seed)
    return venv, raw_holder["env"]


_LOAD = {"DQN": DQN, "PPO": PPO, "A2C": A2C}

def render_models_side_by_side(stage: str, seed_eval: int = 999,
                                seed_model: int = 42,
                                max_steps: int = 3000, fps: int = 15,
                                save_path: Path | None = None) -> list:
    """Carrega DQN, PPO e A2C (mesmo stage/seed_model) e renderiza em paralelo."""
    cv2 = _ensure_cv2()
    venvs, raws, models = {}, {}, {}
    for algo in ALGOS:
        path = MODELS_DIR / algo / f"{algo}_{stage}_{seed_model}.zip"
        if not path.exists():
            print(f"  ⚠ {path.name} não existe — pulando {algo}")
            continue
        venv, raw = _build_env_for_render(stage, seed_eval)
        venvs[algo] = venv; raws[algo] = raw
        models[algo] = _LOAD[algo].load(path, device=DEVICE)

    if not models:
        print("(sem modelos disponíveis)"); return []

    obs_dict = {a: venvs[a].reset() for a in models}
    done_dict = {a: False for a in models}
    composite_frames = []

    for _ in range(max_steps):
        panels = []
        for algo in ALGOS:
            if algo not in models:
                # placeholder cinza
                panels.append(np.full((240, 256, 3), 60, dtype=np.uint8))
                continue
            if done_dict[algo]:
                f = raws[algo].render()
            else:
                action, _ = models[algo].predict(obs_dict[algo], deterministic=True)
                obs_dict[algo], _r, dn, _i = venvs[algo].step(action)
                if dn[0]:
                    done_dict[algo] = True
                f = raws[algo].render()
            if f is None:
                f = np.zeros((240, 256, 3), dtype=np.uint8)
            # label no topo
            f = f.copy()
            cv2.putText(f, algo, (8, 20), cv2.FONT_HERSHEY_DUPLEX, 0.7,
                        (255, 255, 255), 1, cv2.LINE_AA)
            panels.append(f)

        # Empilha lado a lado (h,w,3) → (h, 3w, 3)
        # Normaliza heights
        h = max(p.shape[0] for p in panels)
        panels = [cv2.resize(p, (p.shape[1], h)) if p.shape[0] != h else p
                  for p in panels]
        composite = np.concatenate(panels, axis=1)
        composite_frames.append(composite)

        if all(done_dict.values()):
            break

    for v in venvs.values():
        try: v.close()
        except Exception: pass

    print(f"✓ {len(composite_frames)} frames compostos")
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        imageio.mimsave(save_path, composite_frames,
                        duration=int(round(1000/fps)), loop=0)
        print(f"✓ GIF salvo: {save_path}")
    return composite_frames


# Exemplo (descomente para gerar):
# frames = render_models_side_by_side("1-1", seed_eval=999, seed_model=42,
#                                       save_path=VIDS_DIR / "side_by_side_1-1.gif")
print("✓ render_models_side_by_side() definido")


## 9. Evolução temporal (checkpoints intermediários)

In [ ]:
# --- 9. Evolução temporal: MESMO algoritmo, checkpoints diferentes ---
# Demonstra qualitativamente a aprendizagem. Útil para a apresentação.

def _checkpoint_files(algo: str, stage: str, seed: int) -> list:
    ckpt_dir = MODELS_DIR / "checkpoints" / algo / f"{algo}_{stage}_{seed}"
    if not ckpt_dir.exists():
        return []
    files = list(ckpt_dir.glob(f"{algo}_{stage}_{seed}_*_steps.zip"))
    # ordena por timesteps
    def _n(p):
        try: return int(p.stem.split("_")[-2])
        except: return -1
    return sorted([f for f in files if _n(f) > 0], key=_n)


def render_temporal_evolution(algo: str, stage: str, seed: int,
                               seed_eval: int = 999, max_steps: int = 2000,
                               fps: int = 15, save_path: Path | None = None):
    """Renderiza o mesmo agente em ~3 checkpoints (início, meio, fim) lado a lado."""
    # cv2 self-contained (não depende de _ensure_cv2 ter sido executado antes)
    try:
        import cv2
    except ImportError:
        import subprocess
        subprocess.run("pip install --quiet opencv-python-headless",
                       shell=True, check=True)
        import cv2

    ckpts = _checkpoint_files(algo, stage, seed)
    if not ckpts:
        print(f"(sem checkpoints para {algo}_{stage}_{seed})")
        return []

    # Pega 3 marcos: primeiro, meio, último (ou todos se ≤3)
    if len(ckpts) >= 3:
        ckpts = [ckpts[0], ckpts[len(ckpts)//2], ckpts[-1]]
    labels = [f"{c.stem.split('_')[-2]} steps" for c in ckpts]

    models, venvs, raws = [], [], []
    for c in ckpts:
        venv, raw = _build_env_for_render(stage, seed_eval)
        models.append(_LOAD[algo].load(c, device=DEVICE))
        venvs.append(venv); raws.append(raw)
    obs_list = [v.reset() for v in venvs]
    done_list = [False] * len(models)

    composite_frames = []
    for _ in range(max_steps):
        panels = []
        for i, (m, v, raw, label) in enumerate(zip(models, venvs, raws, labels)):
            if done_list[i]:
                f = raw.render()
            else:
                a, _ = m.predict(obs_list[i], deterministic=True)
                obs_list[i], _r, dn, _info = v.step(a)
                if dn[0]: done_list[i] = True
                f = raw.render()
            if f is None:
                f = np.zeros((240, 256, 3), dtype=np.uint8)
            f = f.copy()
            cv2.putText(f, label, (8, 20), cv2.FONT_HERSHEY_DUPLEX, 0.6,
                        (255, 255, 255), 1, cv2.LINE_AA)
            panels.append(f)
        composite_frames.append(np.concatenate(panels, axis=1))
        if all(done_list): break

    for v in venvs:
        try: v.close()
        except Exception: pass

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        imageio.mimsave(save_path, composite_frames,
                        duration=int(round(1000/fps)), loop=0)
        print(f"✓ GIF salvo: {save_path}")
    return composite_frames


# Exemplo (descomente):
# render_temporal_evolution("PPO", "1-1", 42,
#                            save_path=VIDS_DIR / "temporal_PPO_1-1_42.gif")
print("✓ render_temporal_evolution() definido")


## 10. Onde estão as saídas

Tudo que este notebook gera fica em `BASE_DIR/figures/` e `BASE_DIR/videos/`:

**Tabelas (CSV):**
- `figures/table_group1.csv` — métricas comparativas (R_final, AUC, t80, σ_final)
- `figures/table_group2.csv` — métricas de dificuldade (τ, d̄, mortes, tempo)
- `figures/spearman.csv` — ρ de Spearman para validar H1
- `figures/mann_whitney.csv` — comparação pareada de algoritmos

**Figuras (PNG):**
- `figures/learning_curves.png` — curvas de aprendizado (Fig. 1 do paper)
- `figures/group1_bars.png` — boxplots Grupo I
- `figures/group2_boxplots.png` — boxplots Grupo II

**Vídeos (GIF):**
- `videos/side_by_side_{stage}.gif` — DQN×PPO×A2C jogando lado a lado
- `videos/temporal_{algo}_{stage}_{seed}.gif` — evolução do mesmo agente

**Próximos passos:**
1. Importar as PNGs no `.tex` do paper (subseções 4.x e 5.x da Parte II)
2. Citar os valores das tabelas no texto (mediana ± IQR)
3. Reportar Spearman ρ + p-value no texto: validação da H1
4. Reportar Mann-Whitney com asteriscos de significância nas figuras
5. Subir os GIFs no repositório do paper (suplementar)
